In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # add project root so src is importable

import pandas as pd
import numpy as np 
from src.config import LOANS_RAW

In [3]:
raw= pd.read_csv(LOANS_RAW,skipinitialspace=True)
loan=raw.copy()
loan.columns=loan.columns.str.strip()
loan.head()

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted
0,L500000,C107412,01/01/2025,nano_loan,23638.0,3,20.5,0.22,False
1,L500001,C106708,2025-05-23,nano_loan,23807.0,1,23.0,0.38,False
2,L500002,C106123,25/10/2024,merchant_advance,277761.0,1,34.2,21.70,True
3,L500003,C112525,15/08/2024,nano_loan,12857.0,3,23.9%,0.48,False
4,L500004,C104772,2024-10-22,nano_loan,22631.0,3,21.7%,0.75,False


In [4]:
loan.columns

Index(['loan_id', 'customer_id', 'disbursed_date', 'purpose', 'amount_pkr',
       'term_months', 'interest_rate_pct', 'inflow_to_loan_ratio',
       'defaulted'],
      dtype='str')

In [5]:
# Is loan_id unique? Does customer_id repeat?
# What's in defaulted and purpose (the categoricals)?
# Any impossible numerics (negative amounts, zero terms, wild interest rates)?
# Missing values, and where?

loan.nunique()

loan_id                 8000
customer_id             8000
disbursed_date           646
purpose                    4
amount_pkr              7572
term_months                4
interest_rate_pct        362
inflow_to_loan_ratio    1233
defaulted                  2
dtype: int64

In [6]:
loan.describe().T

,count,mean,std,min,25%,50%,75%,max
amount_pkr,8050.0,68785.590683,94416.994308,-373511.00,12951.25,23206.50,83338.5000,399599.00
term_months,8050.0,3.895652,3.235379,1.00,1.00,3.00,6.0000,12.00
inflow_to_loan_ratio,8050.0,2.505791,4.349396,0.01,0.34,0.82,2.5175,49.05


In [7]:
loan.info()

<class 'pandas.DataFrame'>
RangeIndex: 8050 entries, 0 to 8049
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   loan_id               8050 non-null   str    
 1   customer_id           8050 non-null   str    
 2   disbursed_date        8050 non-null   str    
 3   purpose               8050 non-null   str    
 4   amount_pkr            8050 non-null   float64
 5   term_months           8050 non-null   int64  
 6   interest_rate_pct     8050 non-null   str    
 7   inflow_to_loan_ratio  8050 non-null   float64
 8   defaulted             8050 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(5)
memory usage: 1022.1 KB


In [8]:
loan.value_counts()

loan_id  customer_id  disbursed_date  purpose           amount_pkr  term_months  interest_rate_pct  inflow_to_loan_ratio  defaulted
L500713  C113509      2024-12-24      device_finance    45192.0     12           23.0               1.73                  False        2
L500721  C101801      2024-10-07      nano_loan         5522.0      6            24.7               0.12                  False        2
L500739  C109926      2024-08-10      nano_loan         23718.0     12           20.1               0.32                  True         2
L500948  C111284      2025-02-27      device_finance    65677.0     12           27.1%              1.79                  False        2
L500958  C101465      2024-08-08      nano_loan         13554.0     12           29.5               0.76                  False        2
                                                                                                                                      ..
L507995  C105831      2025-05-26      device_f

In [9]:
# Header padding (fix: skipinitialspace + strip — done)
# Negative loan amounts (count them)
# interest_rate_pct has % signs → stuck as text (strip %, convert)
# 50 duplicate loan_ids (confirm exact vs conflicting)
# disbursed_date as string (check format count)
# inflow_to_loan_ratio extremes (investigate)
# Referential integrity (pending customers join)

dupes= loan[loan.duplicated("loan_id",keep=False)].sort_values("loan_id")
dupes.head() 

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted
713,L500713,C113509,2024-12-24,device_finance,45192.0,12,23.0,1.73,False
8031,L500713,C113509,2024-12-24,device_finance,45192.0,12,23.0,1.73,False
721,L500721,C101801,2024-10-07,nano_loan,5522.0,6,24.7,0.12,False
8025,L500721,C101801,2024-10-07,nano_loan,5522.0,6,24.7,0.12,False
739,L500739,C109926,2024-08-10,nano_loan,23718.0,12,20.1,0.32,True


In [10]:
loan=loan.drop_duplicates(keep="first") 
assert loan.duplicated().sum()==0
assert loan["loan_id"].is_unique

In [11]:
loan.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   loan_id               8000 non-null   str    
 1   customer_id           8000 non-null   str    
 2   disbursed_date        8000 non-null   str    
 3   purpose               8000 non-null   str    
 4   amount_pkr            8000 non-null   float64
 5   term_months           8000 non-null   int64  
 6   interest_rate_pct     8000 non-null   str    
 7   inflow_to_loan_ratio  8000 non-null   float64
 8   defaulted             8000 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(5)
memory usage: 1020.6 KB


In [12]:
loan.head()

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted
0,L500000,C107412,01/01/2025,nano_loan,23638.0,3,20.5,0.22,False
1,L500001,C106708,2025-05-23,nano_loan,23807.0,1,23.0,0.38,False
2,L500002,C106123,25/10/2024,merchant_advance,277761.0,1,34.2,21.70,True
3,L500003,C112525,15/08/2024,nano_loan,12857.0,3,23.9%,0.48,False
4,L500004,C104772,2024-10-22,nano_loan,22631.0,3,21.7%,0.75,False


In [13]:
loan.query("amount_pkr < 0")

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted
2105,L502105,C113768,2025-05-04,merchant_advance,-373511.0,1,25.0,17.96,False
3431,L503431,C106395,2024-09-22,device_finance,-101435.0,1,20.9,2.27,False
4390,L504390,C109619,2024-09-22,merchant_advance,-302449.0,6,23.6,5.02,True


In [14]:
loan["interest_rate_pct"]=loan["interest_rate_pct"].str.strip("% ").astype("float")

In [15]:
loan["purpose"].value_counts()

purpose
nano_loan           3650
merchant_advance    1788
device_finance      1406
emergency           1156
Name: count, dtype: int64

In [16]:
for col in loan.select_dtypes("object").columns:
    loan[col]=loan[col].str.strip()
loan.head()

C:\Program Files\KMSpico\temp\ipykernel_17312\2015923348.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in loan.select_dtypes("object").columns:


,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted
0,L500000,C107412,01/01/2025,nano_loan,23638.0,3,20.5,0.22,False
1,L500001,C106708,2025-05-23,nano_loan,23807.0,1,23.0,0.38,False
2,L500002,C106123,25/10/2024,merchant_advance,277761.0,1,34.2,21.70,True
3,L500003,C112525,15/08/2024,nano_loan,12857.0,3,23.9,0.48,False
4,L500004,C104772,2024-10-22,nano_loan,22631.0,3,21.7,0.75,False


In [17]:
loan["purpose"].value_counts()

purpose
nano_loan           3650
merchant_advance    1788
device_finance      1406
emergency           1156
Name: count, dtype: int64

In [18]:
max=loan["inflow_to_loan_ratio"] > 30
loan[max].head()

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted
914,L500914,C104080,2024-08-22,merchant_advance,332810.0,6,20.2,35.03,False
968,L500968,C111409,2025-03-09,merchant_advance,258447.0,3,33.1,32.31,True
988,L500988,C101693,2024-11-13,merchant_advance,378002.0,1,20.7,30.73,False
1184,L501184,C102515,2025-01-22,merchant_advance,388815.0,6,36.0,34.11,True
1610,L501610,C109736,2025-04-17,merchant_advance,381756.0,1,23.2,47.72,False


In [19]:
q1= loan["inflow_to_loan_ratio"].quantile(0.25)
q3= loan["inflow_to_loan_ratio"].quantile(0.75)

iqr= q3 - q1

lower_side= q1 - 1.5 * iqr
print(lower_side)
upper_side= q3 + 1.5 * iqr

outliers= loan[(loan["inflow_to_loan_ratio"] < lower_side) | (loan["inflow_to_loan_ratio"] > upper_side)]
#outliers["inflow_to_loan_ratio"].describe() 

-2.9300000000000006


In [20]:
loan.nsmallest(20, "inflow_to_loan_ratio")

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted
6503,L506503,C110564,2024-10-26,nano_loan,2134.0,12,33.5,0.01,False
480,L500480,C100129,12/07/2024,nano_loan,2170.0,6,20.8,0.02,False
977,L500977,C111239,2025-03-12,nano_loan,3921.0,3,31.9,0.02,False
1730,L501730,C101067,2025-05-14,nano_loan,2503.0,3,18.6,0.02,False
1994,L501994,C110776,06/09/2024,nano_loan,2258.0,12,23.3,0.02,False
4001,L504001,C108005,2025-01-29,nano_loan,3115.0,1,27.4,0.02,False
4050,L504050,C111158,2025-03-30,nano_loan,2597.0,3,35.2,0.02,False
4232,L504232,C107806,2024-12-20,nano_loan,2377.0,1,19.3,0.02,False
4265,L504265,C103815,2024-09-02,nano_loan,2799.0,6,19.4,0.02,False
6231,L506231,C101108,09/09/2024,nano_loan,2754.0,1,24.3,0.02,False


In [21]:
loan["inflow_to_loan_ratio"].describe()

count    8000.000000
mean        2.509914
std         4.358680
min         0.010000
25%         0.340000
50%         0.820000
75%         2.520000
max        49.050000
Name: inflow_to_loan_ratio, dtype: float64

In [22]:
loan["disbursed_date"].sample(20)

2297    2024-10-21
3301    03/01/2025
2219    2025-05-08
7693    2024-12-03
6280    2025-01-26
4916    2025-05-04
5429    2024-09-02
7021    2024-11-01
6881    07/05/2025
1828    2024-07-29
1813    2024-07-15
1224    2024-08-24
4665    2025-01-24
4773    2025-04-01
3132    14/12/2024
1469    2024-09-09
7972    14/11/2024
4866    2024-10-08
5626    2025-03-30
6193    2025-04-20
Name: disbursed_date, dtype: str

In [23]:
attempt= pd.to_datetime(loan["disbursed_date"].str.strip(), format= "%Y-%m-%d", errors="coerce")
print(attempt.notna().sum())

6800


In [24]:
attempt= pd.to_datetime(loan["disbursed_date"].str.strip(), format= "%d/%m/%Y", errors="coerce")
print(attempt.notna().sum()) 

1200


In [25]:
loan.isna().sum()  

loan_id                 0
customer_id             0
disbursed_date          0
purpose                 0
amount_pkr              0
term_months             0
interest_rate_pct       0
inflow_to_loan_ratio    0
defaulted               0
dtype: int64

In [26]:
slash=pd.to_datetime(loan["disbursed_date"], format="%d/%m/%Y", errors="coerce")
dash=pd.to_datetime(loan["disbursed_date"], format= "%Y-%m-%d", errors="coerce")

merge_date= slash.combine_first(dash)

In [27]:
loan["disbursed_date"]=merge_date

In [28]:
loan["disbursed_date"].isna().sum()

np.int64(0)

In [29]:
loan["purpose"].value_counts()

purpose
nano_loan           3650
merchant_advance    1788
device_finance      1406
emergency           1156
Name: count, dtype: int64

In [30]:
allowed=["nano_loan","merchant_advance","device_finance","emergency"]

assert loan["purpose"].isin(allowed).all()

In [31]:
loan["defaulted"].value_counts()

defaulted
False    6888
True     1112
Name: count, dtype: int64

In [32]:
loan["amount_suspect"] = loan["amount_pkr"] < 0

In [33]:
loan.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   loan_id               8000 non-null   str           
 1   customer_id           8000 non-null   str           
 2   disbursed_date        8000 non-null   datetime64[us]
 3   purpose               8000 non-null   str           
 4   amount_pkr            8000 non-null   float64       
 5   term_months           8000 non-null   int64         
 6   interest_rate_pct     8000 non-null   float64       
 7   inflow_to_loan_ratio  8000 non-null   float64       
 8   defaulted             8000 non-null   bool          
 9   amount_suspect        8000 non-null   bool          
dtypes: bool(2), datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 717.5 KB


In [34]:
loan.head()

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted,amount_suspect
0,L500000,C107412,2025-01-01,nano_loan,23638.0,3,20.5,0.22,False,False
1,L500001,C106708,2025-05-23,nano_loan,23807.0,1,23.0,0.38,False,False
2,L500002,C106123,2024-10-25,merchant_advance,277761.0,1,34.2,21.70,True,False
3,L500003,C112525,2024-08-15,nano_loan,12857.0,3,23.9,0.48,False,False
4,L500004,C104772,2024-10-22,nano_loan,22631.0,3,21.7,0.75,False,False


In [35]:
loan.describe().T

,count,mean,min,25%,50%,75%,max,std
disbursed_date,8000,2024-12-15 12:58:19.200000,2024-07-01 00:00:00,2024-09-23 00:00:00,2024-12-17 00:00:00,2025-03-10 00:00:00,2025-05-26 00:00:00,NaN
amount_pkr,8000.0,68836.59975,-373511.0,12948.0,23200.5,83347.5,399599.0,94537.088848
term_months,8000.0,3.89625,1.0,1.0,3.0,6.0,12.0,3.235474
interest_rate_pct,8000.0,26.943387,18.0,22.4,26.9,31.4,36.0,5.217332
inflow_to_loan_ratio,8000.0,2.509914,0.01,0.34,0.82,2.52,49.05,4.35868


In [36]:
loan["purpose"].astype("category")

0              nano_loan
1              nano_loan
2       merchant_advance
3              nano_loan
4              nano_loan
              ...       
7995      device_finance
7996    merchant_advance
7997           nano_loan
7998           emergency
7999    merchant_advance
Name: purpose, Length: 8000, dtype: category
Categories (4, str): ['device_finance', 'emergency', 'merchant_advance', 'nano_loan']

In [37]:
# ── Step 13: validation gate (src/cleaning/loans.py) ─────────────
from src.config import (
    LOAN_PURPOSES, TERM_MIN, TERM_MAX,
    INTEREST_MIN, INTEREST_MAX, AS_OF_DATE,
)

REQUIRED = [
    "loan_id", "customer_id", "disbursed_date", "purpose", "amount_pkr",
    "term_months", "interest_rate_pct", "inflow_to_loan_ratio", "defaulted",
]

def validate_loans(df):
    # key integrity
    assert df["loan_id"].is_unique, "loan_id not unique"

    # categorical membership
    assert df["purpose"].isin(LOAN_PURPOSES).all(), "unexpected purpose value"

    # amount positive — but exempt the parked negatives, which ride in flagged
    assert (df.loc[~df["amount_suspect"], "amount_pkr"] > 0).all(), \
        "non-flagged amount_pkr <= 0"

    # numeric ranges — legal bounds, not this sample's min/max
    assert df["term_months"].between(TERM_MIN, TERM_MAX).all(), "term out of range"
    assert df["interest_rate_pct"].between(INTEREST_MIN, INTEREST_MAX).all(), \
        "interest out of range"
    assert (df["inflow_to_loan_ratio"] > 0).all(), "ratio must be positive"

    # no future disbursements
    assert (df["disbursed_date"] <= AS_OF_DATE).all(), "disbursed_date in the future"

    # completeness
    assert df[REQUIRED].notna().all().all(), "NaN in a required column"

    # NOTE: referential integrity (customer_id ∈ customers) is deliberately NOT here —
    # it needs the merge, so it lives in the reconciliation file.

    return True